In [62]:
import numpy as np
import pandas as pd
import os

## Configuration

In [63]:
dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"] # DTD | EuroSAT | GTSRB | MNIST | RESISC45 | Stanford_Cars | SUN397 | SVHN
said = 7
domain = "Base_Fine_Tuned" # Base_Fine_Tuned | Fine_Tuned_Layer_Skipping
domain_type = "Long_Training"
model_name = "CLIP_ViT_Vision" # DeiT | CLIP_ViT_Vision | Google_ViT
transformation = ["Standard", "Base_Fine_Tuned_Classifier", "Base_Linear_Probe"] # "Standard" | "Base_Fine_Tuned_Classifier" | "Base_Linear_Probe"
results_path = f"../Data/{domain_type}/{dataset_name[said]}_{domain}" # /Entire_Transformation_Matrix_W"
indices = [i for i in range(12)]
size = [i for i in range(1,6)]

## Loading Data

In [64]:
results = {}

for i in size:
    results[i] = []
    path = f"{results_path}/{i}/Entire_Transformation_Matrix_W"
    try:
        for filename in os.listdir(path):
            if filename in [".DS_Store", f"Base_Fine_Tuned_Classifier_Results_{i}.json", f"Base_Linear_Probe_Results_{i}.json", ".ipynb_checkpoints"]:
                continue
            file_path = os.path.join(path, filename)
            if os.path.isfile(file_path):
                results[i].append(file_path)
    except FileNotFoundError:
        print(f"Error: The Folder '{path}' was not found.")
    except Exception as e:
        print(f"An error occured: {e}")

    data = [pd.read_json(i) for i in results[i]]
    data = sorted(data, key=lambda df: df["Train_Data_Size"][0])
    results[i] = data
    print(len(results[i]))

30
30
30
30
30


## Finding Best Numbers

In [65]:
def find_best_acc(data):
    acc = {i: [] for i in indices} # Indices, then len(data)
    for i in indices:
        for j in range(len(data)):
            acc[i].append(data[j]["Classification_Accuracy"][i])

    best_acc = {}

    for i in acc:
        arr = acc[i]
        max_val = max(arr)
        indice = arr.index(max_val)
        num_img = data[indice]["Train_Data_Size"][i][0]
        best_acc[i] = (max_val, num_img)

    final = []
    num_img = []
    for i in indices:
        final.append(best_acc[i][0])
        num_img.append(best_acc[i][1])

    best_accuracy = max(final)
    index = final.index(best_accuracy)
    best_accuracy_num_images = num_img[index]

    return best_accuracy, best_accuracy_num_images, index

In [66]:
best_acc = []
num_img = []
index = []
print(f"{model_name} - {dataset_name[said]}: {domain}")
for i in size:
    acc, img, ind = find_best_acc(results[i])
    best_acc.append(acc)
    num_img.append(img)
    index.append(ind)
    print(f"Set {i} Best Accuracy: {best_acc[i-1]} | Number of training images: {num_img[i-1]} | Transformation Layer (0-11): {index[i-1]}")
print(f"Average: {np.mean(best_acc)} +- {np.var(best_acc)}")

CLIP_ViT_Vision - SVHN: Base_Fine_Tuned
Set 1 Best Accuracy: 0.6683312846 | Number of training images: 73257 | Transformation Layer (0-11): 8
Set 2 Best Accuracy: 0.6675245851 | Number of training images: 73257 | Transformation Layer (0-11): 8
Set 3 Best Accuracy: 0.6601874616 | Number of training images: 58605 | Transformation Layer (0-11): 8
Set 4 Best Accuracy: 0.6674861709000001 | Number of training images: 58605 | Transformation Layer (0-11): 8
Set 5 Best Accuracy: 0.6761293792 | Number of training images: 58605 | Transformation Layer (0-11): 8
Average: 0.66793177628 +- 2.553981584127224e-05
